# MoshiRAG retrieval-latency sweep on a Colab T4 (low-memory version)

Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Every step is resumable:
if Colab disconnects, rerun the setup cells (0–5) and then the step you were on — completed runs are skipped.

What runs: MoshiRAG (int8-quantized LM, float16), the same frame-based retrieval-latency injection as the
A100 version, a fixed reference whose embedding is precomputed once. No STT, no retrieval LLM.
See `experiments/async_retrieval_latency/t4/README.md`.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Settings
Results and the embedding cache go to Google Drive so they survive disconnects.
The Hugging Face cache (~18 GB, or ~30 GB with the encoder) stays on the local disk.

In [ ]:
REPO_URL = "https://github.com/zxx3312/moshi-rag.git"   # your fork
BRANCH = "exp/async-retrieval-latency"
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/moshirag_t4"
else:
    WORK = "/content/moshirag_t4"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_home"   # set before anything imports huggingface_hub
REPO = "/content/moshi-rag"
EXP = f"{REPO}/experiments/async_retrieval_latency"
CACHE = f"{WORK}/reference_cache"
RESULTS = f"{WORK}/results"
print(WORK, CACHE, RESULTS, sep="\n")

## 2. Clone the repository (private fork: use a GitHub token in the URL or `gh auth login`)

In [ ]:
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
!git -C {REPO} log --oneline -1

## 3. Install pinned dependencies (2–4 min; nothing here imports torch in this notebook, so no restart is needed)

In [ ]:
!pip install -q -r {EXP}/t4/requirements-colab.txt

## 4. Hugging Face login
Needed only for step 6 (the reference encoder's tokenizer comes from the gated
`meta-llama/Llama-3.2-3B-Instruct`; only its ~9 MB tokenizer files are downloaded, never the Llama weights).
Use a **read** token whose account has accepted the Llama 3.2 license.

In [ ]:
from huggingface_hub import login
login()

## 5. Make the question WAVs (Piper TTS in a separate venv, CPU, ~1 s per question)

In [ ]:
import json, pathlib, subprocess
if not os.path.exists("/content/tts/bin/python"):
    !python -m venv /content/tts && /content/tts/bin/pip install -q piper-tts numpy
    !/content/tts/bin/python -m piper.download_voices en_US-lessac-medium --data-dir /content/piper_voices
for manifest in [f"{EXP}/configs/samples.jsonl", f"{EXP}/t4/configs/samples_small.jsonl"]:
    for line in open(manifest):
        s = json.loads(line)
        wav = (pathlib.Path(manifest).parent / s["wav"]).resolve()
        if not wav.exists():
            subprocess.run(["/content/tts/bin/python", f"{EXP}/scripts/make_tts_question.py",
                            "--voice", "/content/piper_voices/en_US-lessac-medium.onnx",
                            "--text", s["question"], "--out", str(wav)], check=True)
!ls -la {EXP}/configs/audio

## 6. Precompute reference embeddings (one time; separate process, frees the GPU when it exits)
Downloads the MoshiRAG checkpoint (15.4 GB) and the ARC encoder (12.1 GB). On a T4 the encoder autocasts to
float16 (no bf16 support); the official encoder uses bfloat16. For exact parity, compute the cache on an
A100 instead and copy it to `CACHE`.

In [ ]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples.jsonl {EXP}/t4/configs/samples_small.jsonl --cache-dir {CACHE}

## 7. Step 1 — 1 sample × 1 delay (3 s)
Checks: model loads in 16 GB, `[RET]` → injection works, audio + JSONL are written. Read the memory lines in the output.

In [ ]:
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS}

In [ ]:
import json
run = f"{RESULTS}/step1_1x1"
print(open(f"{run}/run_status.json").read())
for r in map(json.loads, open(f"{run}/results.jsonl")):
    print({k: r[k] for k in ["run_id", "status", "stop_reason", "rag_triggered", "retrieval_trigger_time",
                             "reference_injection_time", "trigger_to_injection_s", "gpu_peak_allocated_mb", "run_wall_s"]})
    print("pre-RAG :", r["pre_rag_transcript"]); print("post-RAG:", r["post_rag_transcript"])
from IPython.display import Audio
Audio(f"{run}/audio/" + r["run_id"] + "_stereo.wav")

## 8. Step 2 — 1 sample × all delays (0, 1, 2, 3, 5, 8 s)

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --run-name step2_1xall {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step2_1xall

## 9. Step 3 — small multi-sample sweep (5 samples × 6 delays = 30 runs)
Long: rerun this cell after a disconnect, it resumes. The 5 questions are examples; choose your own evaluation set.

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/t4/configs/samples_small.jsonl --run-name step3_small {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step3_small

## GPU memory log

In [ ]:
for line in open(f"{RESULTS}/step1_1x1/logs/gpu_memory.jsonl"):
    r = json.loads(line)
    print(f"{r['event']:28s} allocated {r.get('allocated_mb', 0):8.0f} MB  peak {r.get('peak_allocated_mb', 0):8.0f} MB  device {r.get('device_used_mb', 0):8.0f} MB")